In [ ]:
# IMPORT LIBRARIES
import pandas as pd
import numpy as np

In [9]:
import os
print(os.listdir("../data/raw"))

['2019-Nov.csv']


In [10]:
import pandas as pd

df = pd.read_csv("D:/customer-segmentation-unsupervised/data/raw/2019-Nov.csv/2019-Nov.csv", nrows=500000)

print("Loaded Successfully ")
print(df.shape)
df.head()

Loaded Successfully 
(500000, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [11]:
# BASIC INFORMATION
print("Shape:", df.shape)
df.info()

# Check missing values
print("\nMissing Values:\n", df.isnull().sum())

# Check unique event types
print("\nEvent Types:", df["event_type"].unique())

Shape: (500000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     500000 non-null  str    
 1   event_type     500000 non-null  str    
 2   product_id     500000 non-null  int64  
 3   category_id    500000 non-null  int64  
 4   category_code  337628 non-null  str    
 5   brand          425209 non-null  str    
 6   price          500000 non-null  float64
 7   user_id        500000 non-null  int64  
 8   user_session   500000 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 34.3 MB

Missing Values:
 event_time            0
event_type            0
product_id            0
category_id           0
category_code    162372
brand             74791
price                 0
user_id               0
user_session          0
dtype: int64

Event Types: <StringArray>
['view', 'cart', 'purchase']
Length: 3, dtype: str


In [12]:
# DATA CLEANING
# Remove missing values (important columns)
df = df.dropna(subset=["user_id", "price"])

# Convert event_time to datetime
df["event_time"] = pd.to_datetime(df["event_time"])

# Remove invalid prices
df = df[df["price"] > 0]

# Remove duplicates
df = df.drop_duplicates()

print("After Cleaning Shape:", df.shape)

After Cleaning Shape: (499202, 9)


In [13]:
# FEATURE EXTRACTION
# Extract date features
df["date"] = df["event_time"].dt.date
df["hour"] = df["event_time"].dt.hour
df["day_of_week"] = df["event_time"].dt.day_name()

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,date,hour,day_of_week
0,2019-11-01 00:00:00+00:00,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33,2019-11-01,0,Friday
1,2019-11-01 00:00:00+00:00,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283,2019-11-01,0,Friday
2,2019-11-01 00:00:01+00:00,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387,2019-11-01,0,Friday
3,2019-11-01 00:00:01+00:00,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f,2019-11-01,0,Friday
4,2019-11-01 00:00:01+00:00,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2,2019-11-01,0,Friday


In [14]:
# FILTER EVENTS
df = df[df["event_type"].isin(["view", "cart", "purchase"])]

print(df["event_type"].value_counts())

event_type
view        482054
purchase      9595
cart          7553
Name: count, dtype: int64


In [15]:
# CHECK BASIC STATS
print(df.describe())

         product_id   category_id          price       user_id           hour
count  4.992020e+05  4.992020e+05  499202.000000  4.992020e+05  499202.000000
mean   1.063644e+07  2.057479e+18     289.849953  5.351774e+08       5.395643
std    1.190526e+07  1.879533e+16     348.147844  2.010915e+07       2.213232
min    1.000978e+06  2.053014e+18       0.770000  2.749691e+08       0.000000
25%    1.005196e+06  2.053014e+18      68.210000  5.159156e+08       4.000000
50%    5.100511e+06  2.053014e+18     167.310000  5.308112e+08       6.000000
75%    1.570011e+07  2.053014e+18     360.340000  5.542951e+08       7.000000
max    6.050001e+07  2.180737e+18    2574.070000  5.663943e+08       9.000000


In [16]:
# SAVE CLEAN DATA
output_path = "D:/customer-segmentation-unsupervised/data/processed/clean_data.csv"

df.to_csv(output_path, index=False)

print("Clean data saved successfully ")

Clean data saved successfully 
